# C-MAPSS RUL: thin orchestration notebook

**All logic lives in `src/` — this notebook only installs, imports, overrides config, calls functions, and plots.**
See `RESEARCH_PLAN.md` for scope and protocol, and `CHANGES.md` for every deviation.

## How to run (in order)
1. **Setup** — installs + mount Drive + `sys.path` + GPU check.
2. **Config** — point paths at your Drive; override any design decision here.
3. **Stage A (GPU, once per embedding config)** — Chronos-2 `embed()` → versioned cache on Drive (pooled embeddings **+ per-window loc/scale + fixed raw windows**). Variable-length TSFM contexts are fed to `embed()` natively (short test histories are left-pad-**masked**, not fabricated). *Idempotent*: re-running skips if the cache key exists. Degrades to a T4 via `embed_batch_size`/`embed_dtype`.
4. **Stage A2 (ablation)** — full-data, MSE, 3 seeds over `tsfm_context_length × head_features` (+ the raw-fusion arm and pooling variants at the best cell). Picks the winning **(context, features, pooling)** cell; builds one Stage A cache per context/pooling as needed.
5. **Stage B (cheap, rerunnable)** — data-fraction × loss × seed sweep + baselines at the **winning** config, on the **cached** arrays. Writes `results_v2.csv` with **both** test-label protocols (clipped + unclipped). Checkpoints every cell; never re-embeds.
6. **Stage C** — load result CSVs and plot the learning curves + the data-scaling curve.

The economics: **embeddings once per config (Stage A), everything downstream free (Stages A2/B).**

## 1. Setup

In [ ]:
# Pinned installs. Colab already ships torch/numpy/pandas/scikit-learn/matplotlib.
# chronos-forecasting 2.x provides the official Chronos2Pipeline.embed().
%pip install -q "chronos-forecasting>=2.0.0" "coral-pytorch==1.4.0" "lightgbm>=4.0" "sktime>=1.0" "numba>=0.59" "safetensors>=0.4"

import sys, os, torch
from google.colab import drive

# Mount Drive (holds CMAPSSData/, the embedding cache, and results).
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Add the repo (kept on Drive) to sys.path so `import src.*` works.
REPO_DIR = '/content/drive/MyDrive/Predictive Maintenance LSTM/Predictive-Maintenance-LSTM'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Config

The single source of truth is `src/config.py`. Override only paths (and any design decision you want to ablate) here.

In [ ]:
from src.config import Config

DRIVE = '/content/drive/MyDrive/Predictive Maintenance LSTM'
config = Config(
    dataset='FD001',
    data_dir=f'{DRIVE}/CMAPSSData',
    cache_dir=f'{DRIVE}/cache',
    results_dir=f'{DRIVE}/results',
    # --- ablation / GPU knobs (all optional; Stage A2 sweeps these for you) ---
    # tsfm_context_length=120,          # TSFM history, independent of window_size (None => window_size)
    # head_features='emb+locscale',     # emb | emb+locscale | emb+locscale+raw
    # pooling='mean',                   # forecast_token | last_content | mean | flatten
    # embed_batch_size=64,              # lower for a T4
    # embed_dtype='float16',            # embed() compute dtype: bfloat16 | float16 | float32
    # embedding_storage_dtype='float32',# on-disk emb dtype (default float16; halves Drive I/O)
    # losses=['mse', 'corn', 'quantile'],
)
config

## 3. Stage A — Embedding pass (GPU, run once per embedding config)

Loads FD001, builds **fixed** baseline windows (for the baselines + the raw-fusion last-cycle sensors) **and** variable-length TSFM contexts aligned 1:1 to them, runs Chronos-2 `embed()` batched on GPU, and caches: pooled embeddings (**float16**), per-window **loc/scale** (float32), fixed raw windows (float32), labels, unit IDs. Keyed by `config.embedding_cache_key()` (window size, pooling, model, **tsfm_context_length**, **schema version**). **Idempotent** — if the key exists it loads instead of recomputing. Throughput (windows/s) is printed and stored in the sidecar.

> Running the *ablation* (Stage A2) next builds these caches for you per (context, pooling). Run this cell directly only for a single explicit config.

In [ ]:
from src.embeddings import build_embedding_cache

cache_path = build_embedding_cache(config)   # skips instantly if the key already exists
print('embedding cache:', cache_path)
print('sidecar        :', cache_path.with_suffix('.json').read_text())

## 3b. Stage A2 — Ablation (pick the winning config)

Full-data, MSE, 3 seeds over `tsfm_context_length ∈ {30,60,120,256} × head_features ∈ {emb, emb+locscale}`, then the `emb+locscale+raw` arm and the `{mean, last_content}` pooling variants at the best cell. Each (context, pooling) triggers one Stage A cache build (idempotent). Restartable — rows are checkpointed to `ablation.csv`. `select_best_ablation_cell` picks the winner by **clipped RMSE** (the literature-comparable metric). Record the winner + justification in `CHANGES.md` §9.

In [ ]:
from src.sweep import run_ablation, select_best_ablation_cell
import torch, pandas as pd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ablation_csv = run_ablation(config, device=device)   # builds caches + trains heads over the grid

best = select_best_ablation_cell(ablation_csv)        # winner by clipped RMSE
print('WINNING CELL:', best)

# Full ablation table (mean ± std clipped RMSE over seeds), sorted best-first.
abl = pd.read_csv(ablation_csv)
summary = (abl.groupby(['tsfm_context_length', 'head_features', 'pooling'])['rmse_clipped']
              .agg(['mean', 'std', 'count']).sort_values('mean'))
summary

## 4. Stage B — Sweep at the winning config (cheap, rerunnable)

Data-fraction × loss × seed MLP heads at the **winning** `(context, head_features, pooling)` from Stage A2, plus baselines on the cached raw windows. Consumes the cache only — **no re-embedding**. Writes `results_v2.csv` with **both** test-label protocols (`rmse_clipped`/`…_unclipped`); any pre-existing `results.csv` is archived to `results_v1.csv` first. Appends after every cell and skips completed cells on restart.

In [ ]:
from src.sweep import run_sweep, run_baseline_window_comparison
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'   # heads train on-GPU (tensor slicing)

# Adopt the ablation winner. (Also builds its Stage A cache if not already present.)
sweep_config = config.replace(
    tsfm_context_length=int(best['tsfm_context_length']),
    head_features=best['head_features'],
    pooling=best['pooling'],
)
from src.embeddings import build_embedding_cache
build_embedding_cache(sweep_config)                       # idempotent; no-op if Stage A2 built it

# Optional (Task 1.5): compare GBM/LSTM at window 30 vs 120 at full data. If a longer
# window wins, adopt it per baseline, e.g. sweep_config = sweep_config.replace(
#     baseline_windows={'lstm': 120}).  Then baselines are re-windowed from the raw series.
# run_baseline_window_comparison(config, device=device)

results_csv = run_sweep(sweep_config, device=device)
print('results CSV:', results_csv)

## 5. Stage C — Results

### Data-scaling curve (the headline figure): test error vs. training units, one line per model, error bands over seeds. Plotted for the **clipped** protocol (literature-comparable) and the **unclipped** protocol.

In [ ]:
import matplotlib.pyplot as plt
from src.evaluate import aggregate_data_scaling

# Both test-label protocols + the NASA score. Clipped RMSE is the literature-comparable
# number; the plan's sanity gate is clipped RMSE <= ~14 (<= ~12.5 with raw fusion).
panels = [('rmse_clipped', 'test RMSE (clipped)'),
          ('rmse_unclipped', 'test RMSE (unclipped)'),
          ('nasa_clipped', 'NASA score (clipped)')]
for metric, ylabel in panels:
    agg = aggregate_data_scaling(results_csv, metric=metric)
    plt.figure(figsize=(7, 5))
    for label, (ns, mean, std) in sorted(agg.items()):
        plt.plot(ns, mean, marker='o', label=label)
        plt.fill_between(ns, mean - std, mean + std, alpha=0.15)
    if metric == 'rmse_clipped':
        plt.axhline(14, ls='--', c='gray', lw=1, label='sanity gate (clipped RMSE ~14)')
    plt.xscale('log'); plt.xlabel('training units'); plt.ylabel(ylabel)
    plt.title(f'Data-scaling curve (FD001) — {ylabel}')
    plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

### Learning curves: validation RMSE vs. epoch, per sweep cell.

In [ ]:
from pathlib import Path
from src.evaluate import load_learning_curve

curve_files = sorted(Path(config.results_dir, 'runs', 'learning_curves').glob('*.csv'))
plt.figure(figsize=(8, 5))
for cf in curve_files:
    c = load_learning_curve(cf)
    xs, ys = c['val_rmse']
    plt.plot(xs, ys, alpha=0.6, label=cf.stem)
plt.xlabel('epoch'); plt.ylabel('val RMSE'); plt.title('Learning curves (val RMSE)')
plt.legend(fontsize=6, ncol=2); plt.grid(alpha=0.3); plt.show()